In [6]:
# Task 3.1 - features, train/test split and a baseline
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_csv("appointments_clean.csv")
X = df[["age", "distance_km", "fee_inr", "lead_days", "prior_no_shows",
        "reminder_sent", "booking_channel", "visit_type"]].copy()
X["reminder_sent"] = X["reminder_sent"].map({"Yes": 1, "No": 0})
X = pd.get_dummies(X, columns=["booking_channel", "visit_type"])
y = df["no_show"]

print("Columns in X:", X.shape[1])
print(list(X.columns))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("\nTraining rows:", len(X_train))
print("Test rows:", len(X_test))
print("No-shows in test set:", y_test.sum())
print("No-show rate overall:", round(y.mean() * 100, 1), "%")

baseline = (y_test == 0).mean()
print("Baseline accuracy:", round(baseline, 4), f"({(y_test == 0).sum()} of {len(y_test)})")

Columns in X: 11
['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel_App', 'booking_channel_Phone', 'booking_channel_Walk-In', 'visit_type_Consultation', 'visit_type_Follow-up']

Training rows: 792
Test rows: 198
No-shows in test set: 49
No-show rate overall: 22.0 %
Baseline accuracy: 0.7525 (149 of 198)


In [7]:
# Task 3.2 - logistic regression
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, precision_score,
                             recall_score, f1_score, roc_auc_score)


scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_s, y_train)

lr_pred = lr.predict(X_test_s)
lr_prob = lr.predict_proba(X_test_s)[:, 1]

print("Accuracy:", round(accuracy_score(y_test, lr_pred), 3))
print("Confusion matrix:\n", confusion_matrix(y_test, lr_pred))
print("Precision:", round(precision_score(y_test, lr_pred), 3))
print("Recall:", round(recall_score(y_test, lr_pred), 3))
print("F1:", round(f1_score(y_test, lr_pred), 3))
print("ROC-AUC:", round(roc_auc_score(y_test, lr_prob), 3))

# coefficients next to the feature names
coefs = pd.DataFrame({"feature": X.columns, "coefficient": lr.coef_[0]})
print("\nCoefficients:")
print(coefs.round(3).to_string(index=False))


i = list(X.columns).index("reminder_sent")
coef = lr.coef_[0][i]
sd = scaler.scale_[i]
print("\nreminder_sent coefficient:", round(coef, 3))
print("np.exp(coef) - per 1 standard deviation:", round(np.exp(coef), 2))
print("standard deviation of reminder_sent:", round(sd, 2))
print("np.exp(coef / sd) - going from No to Yes:", round(np.exp(coef / sd), 2))

# 0.56 is the change in odds for one standard deviation (0.49) of the scaled
# feature, while 0.31 is for the real change from No (0) to Yes (1), which is
# about two standard deviations - that's why the numbers differ.
# In this model, holding the other features constant, appointments with a
# reminder have about 69% lower odds of a no-show. This is an association
# from observational data, not proof that reminders cause fewer no-shows.

Accuracy: 0.788
Confusion matrix:
 [[139  10]
 [ 32  17]]
Precision: 0.63
Recall: 0.347
F1: 0.447
ROC-AUC: 0.789

Coefficients:
                feature  coefficient
                    age       -0.254
            distance_km        0.253
                fee_inr        0.453
              lead_days        0.614
         prior_no_shows        0.650
          reminder_sent       -0.581
    booking_channel_App       -0.047
  booking_channel_Phone        0.216
booking_channel_Walk-In       -0.186
visit_type_Consultation        0.016
   visit_type_Follow-up       -0.016

reminder_sent coefficient: -0.581
np.exp(coef) - per 1 standard deviation: 0.56
standard deviation of reminder_sent: 0.49
np.exp(coef / sd) - going from No to Yes: 0.31


In [8]:
# Task 3.3 - decision trees and overfitting
from sklearn.tree import DecisionTreeClassifier

tree_full = DecisionTreeClassifier(random_state=42)
tree_full.fit(X_train, y_train)
print("Full tree (depth", tree_full.get_depth(), ")")
print("  train accuracy:", round(tree_full.score(X_train, y_train), 3))
print("  test accuracy:", round(tree_full.score(X_test, y_test), 3))

tree3 = DecisionTreeClassifier(max_depth=3, random_state=42)
tree3.fit(X_train, y_train)
print("\nDepth-3 tree")
print("  train accuracy:", round(tree3.score(X_train, y_train), 3))
print("  test accuracy:", round(tree3.score(X_test, y_test), 3))

# The full tree gets 100% on training data but only about 67% on the test set,
# which is even below the 75% baseline - it has memorised noise (overfitting).
# The depth-3 tree has a much smaller gap, so it generalises better.

Full tree (depth 15 )
  train accuracy: 1.0
  test accuracy: 0.672

Depth-3 tree
  train accuracy: 0.822
  test accuracy: 0.778


In [9]:
# Task 3.4 - random forest and a comparison table
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]

print("Random forest")
print("  train accuracy:", round(rf.score(X_train, y_train), 3))
print("  test accuracy:", round(accuracy_score(y_test, rf_pred), 3))
print("  recall:", round(recall_score(y_test, rf_pred), 3))
print("  ROC-AUC:", round(roc_auc_score(y_test, rf_prob), 3))

importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nTop 3 features:")
print(importances.head(3).round(3))

tree3_pred = tree3.predict(X_test)
tree3_prob = tree3.predict_proba(X_test)[:, 1]

comparison = pd.DataFrame({
    "model": ["Logistic regression", "Decision tree (depth 3)", "Random forest"],
    "accuracy": [accuracy_score(y_test, lr_pred), accuracy_score(y_test, tree3_pred), accuracy_score(y_test, rf_pred)],
    "recall": [recall_score(y_test, lr_pred), recall_score(y_test, tree3_pred), recall_score(y_test, rf_pred)],
    "roc_auc": [roc_auc_score(y_test, lr_prob), roc_auc_score(y_test, tree3_prob), roc_auc_score(y_test, rf_prob)],
}).round(3)
print()
print(comparison.to_string(index=False))

# Logistic regression looks best here - it has the highest accuracy, recall
# and ROC-AUC of the three. This table is only for reporting though; the
# actual model choice is made in Task 3.5 using the training data only.

Random forest
  train accuracy: 1.0
  test accuracy: 0.747
  recall: 0.306
  ROC-AUC: 0.706

Top 3 features:
lead_days      0.220
age            0.197
distance_km    0.195
dtype: float64

                  model  accuracy  recall  roc_auc
    Logistic regression     0.788   0.347    0.789
Decision tree (depth 3)     0.778   0.224    0.700
          Random forest     0.747   0.306    0.706


In [10]:
# Task 3.5 - tune, pick the model and threshold, then test once
from sklearn.model_selection import GridSearchCV, cross_val_score, cross_val_predict
from sklearn.pipeline import Pipeline

grid = GridSearchCV(RandomForestClassifier(random_state=42),
                    {"n_estimators": [100, 200], "max_depth": [3, 5, 8]},
                    cv=5, scoring="roc_auc")
grid.fit(X_train, y_train)
print("Best params:", grid.best_params_)
print("Best CV ROC-AUC:", round(grid.best_score_, 3))

lr_pipe = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000))])

cv_scores = {
    "Logistic regression": cross_val_score(lr_pipe, X_train, y_train, cv=5, scoring="roc_auc").mean(),
    "Decision tree (depth 3)": cross_val_score(DecisionTreeClassifier(max_depth=3, random_state=42),
                                               X_train, y_train, cv=5, scoring="roc_auc").mean(),
    "Random forest": cross_val_score(RandomForestClassifier(n_estimators=100, random_state=42),
                                     X_train, y_train, cv=5, scoring="roc_auc").mean(),
    "Tuned random forest": grid.best_score_,
}
print("\nCross-validated ROC-AUC (training set):")
for name, score in cv_scores.items():
    print(f"  {name}: {score:.3f}")
print("Winner:", max(cv_scores, key=cv_scores.get))

print("(info only) tuned forest test ROC-AUC:", round(roc_auc_score(y_test, grid.predict_proba(X_test)[:, 1]), 3))

oof = cross_val_predict(lr_pipe, X_train, y_train, cv=5, method="predict_proba")[:, 1]

print("\nThreshold comparison (training, out-of-fold):")
for t in [0.5, 0.35]:
    flags = (oof >= t).astype(int)
    print(f"  {t}: precision {precision_score(y_train, flags):.3f}, "
          f"recall {recall_score(y_train, flags):.3f}, "
          f"F1 {f1_score(y_train, flags):.3f}, flagged {flags.sum()}")

threshold = 0.35
# 0.35 has the higher F1 and catches far more no-shows. Recall matters here
# because a missed warning means an empty doctor slot, while an extra phone
# call only costs the front desk a minute.


lr_pipe.fit(X_train, y_train)
probs = lr_pipe.predict_proba(X_test)[:, 1]
flag = (probs >= threshold).astype(int)

print("\nFinal test (logistic regression, threshold 0.35):")
print("  precision:", round(precision_score(y_test, flag), 3))
print("  recall:", round(recall_score(y_test, flag), 3))
print("  flagged:", flag.sum())
print("  ROC-AUC:", round(roc_auc_score(y_test, probs), 3))


risk = pd.DataFrame({
    "appointment_id": df.loc[X_test.index, "appointment_id"],
    "no_show": y_test,
    "risk_prob": probs.round(4),
    "flag": flag,
})
risk.to_csv("test_risk_scores.csv", index=False)
print("\ntest_risk_scores.csv saved:", len(risk), "rows,", risk["flag"].sum(), "flagged")

# In practice a clinic would set the threshold by how many follow-up calls
# the front desk can make in an evening, weighed against the cost of an
# empty slot.

Best params: {'max_depth': 3, 'n_estimators': 200}
Best CV ROC-AUC: 0.767

Cross-validated ROC-AUC (training set):
  Logistic regression: 0.774
  Decision tree (depth 3): 0.678
  Random forest: 0.747
  Tuned random forest: 0.767
Winner: Logistic regression
(info only) tuned forest test ROC-AUC: 0.741

Threshold comparison (training, out-of-fold):
  0.5: precision 0.575, recall 0.249, F1 0.347, flagged 73
  0.35: precision 0.513, recall 0.467, F1 0.489, flagged 154

Final test (logistic regression, threshold 0.35):
  precision: 0.571
  recall: 0.571
  flagged: 49
  ROC-AUC: 0.789

test_risk_scores.csv saved: 198 rows, 49 flagged
